In [9]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

In [10]:
data = np.load("../data/moves.npz")

X, y, groups = data["X"], data["y"], data["groups"]

In [11]:
# Check which features are useful:

features_after_board = [
    "side to move", "moving side clock", "base time", "increment",
    "player elo", "opponent elo", "material difference", "number of legal moves",
    "hanging count white", "hanging count black"
]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

train_idx, test_idx = next(gss.split(X, groups=groups))

for i in range(12):
    if i == 0:
        # Clock, ratings, side to move, material difference, number of legal moves, and hanging pieces only features
        temp = X[:, 768:].copy()
    elif i == 1:
        # board only features
        temp = X[:, :768].copy()
    # see what adding each feature does starting after the board features:
    elif i > 1:
        temp = X[:, :768 + (i - 1)].copy()
    else:
        # every feature
        temp = X.copy()

    X_train = temp[train_idx]
    X_test = temp[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = LGBMClassifier(verbosity=-1, random_state=42)

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)
    ap_score = average_precision_score(y_test, y_prob)

    if i == 0:
        print("Clock, ratings, side to move, material difference, number of legal moves, and hanging pieces only features")
        print(f"Roc Auc: {auc_score:0.4f}")
        print(f"Average Precision: {ap_score:0.4f}\n")

    elif i == 1:
        print("Board only features")
        print(f"Roc Auc: {auc_score:0.4f}")
        print(f"Average Precision: {ap_score:0.4f}\n")

    elif 2 <= i < 11:
        print(f"Board only features, {", ".join([feature for feature in features_after_board[:(i-1)]])}")
        print(f"Roc Auc: {auc_score:0.4f}")
        print(f"Average Precision: {ap_score:0.4f}\n")

    else:
        print("Every feature")
        print(f"Roc Auc: {auc_score:0.4f}")
        print(f"Average Precision: {ap_score:0.4f}\n")

Clock, ratings, side to move, material difference, number of legal moves, and hanging pieces only features
Roc Auc: 0.2735
Average Precision: 0.0756

Board only features
Roc Auc: 0.2918
Average Precision: 0.0749

Board only features, side to move
Roc Auc: 0.1490
Average Precision: 0.0633

Board only features, side to move, moving side clock
Roc Auc: 0.1755
Average Precision: 0.0650

Board only features, side to move, moving side clock, base time
Roc Auc: 0.1755
Average Precision: 0.0650

Board only features, side to move, moving side clock, base time, increment
Roc Auc: 0.1755
Average Precision: 0.0650

Board only features, side to move, moving side clock, base time, increment, player elo
Roc Auc: 0.2122
Average Precision: 0.0679

Board only features, side to move, moving side clock, base time, increment, player elo, opponent elo
Roc Auc: 0.2204
Average Precision: 0.0686

Board only features, side to move, moving side clock, base time, increment, player elo, opponent elo, material diff